# Variance explained by interaction order

Decomposes each fitted landscape in the orthogonal Walsh-Hadamard basis, so the cumulative order <= k prediction is the closest order <= k approximation of the full fit. Truncated predictions are censored at the detection floor and scored against the raw per-measurement values with replicates pooled, which is why the attainable ceiling falls below 100 percent and falls further for heavily censored complexes. A complex is included only if its full model explains more than the observed mean and fewer than half its measurements sit at the floor.

## Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple, HandlerPatch

data_dir = "../data"
fig_dir  = "../figures"
os.makedirs(fig_dir, exist_ok=True)

all_antibodies = ["1NC9", "CH235.12", "N49P7", "NIH45-46", "VRC-PG20",
                  "VRC01", "VRC07", "VRC13", "VRC16"]
antigen_order  = ["CH505TF", "SF162"]

# file stems differ from display names: dots and dashes become underscores or vanish
file_prefix = {"1NC9": "1NC9", "CH235.12": "CH235_12", "N49P7": "N49P7",
               "NIH45-46": "NIH4546", "VRC-PG20": "VRCPG20", "VRC01": "VRC01",
               "VRC07": "VRC07", "VRC13": "VRC13", "VRC16": "VRC16"}

# panel membership: None runs the inclusion rule, or hardcode a list to override
panel_antibodies  = None
min_r2_pct        = 0.0    # fail if the full model explains no more than the observed mean
max_censored_frac = 0.50   # fail if more than half of measurements sit at the floor

censor_floor   = 6.0            # detection floor in -log10(Kd)
variance_basis = "measurement"  # "measurement" pools replicates, "genotype" uses replicate means
orders_shown   = None           # None auto-detects the highest fitted order across the panel

antigen_base      = {"CH505TF": "#E66100", "SF162": "#5D3A9B"}
order_lighten     = {1: 0.0, 2: 0.28, 3: 0.50, 4: 0.68, 5: 0.80, 6: 0.88}
unexplained_color = "#c9c9c9"

bar_labels    = "none"   # "all", "none", or "epistatic"
min_seg_label = 4.0      # only label a segment at least this tall, in percent
group_gap     = 0.6      # blank rows between antibody groups
plot_aspect   = 1.0
pct_gridlines = True

fs_axis, fs_tick, fs_header, fs_legend, fs_seg = 15, 11, 13, 12, 9
legend_shade_pad        = 0.0    # 0 puts the order shades flush against each other
legend_unexplained_frac = 0.45   # unexplained swatch width, relative to a full graded strip

fig_stem  = "epistasis_variance_hbar"
fig_dpi   = 300
save_exts = ["png", "pdf", "svg"]
plt.rcParams["svg.fonttype"] = "path"   # outlined text, the only form Illustrator renders here
plt.rcParams["pdf.fonttype"] = 42       # embedded TrueType, so PDF text stays retypeable



## Loading

Resolves the BEST coefficient file per complex and the per-measurement observed affinities. Genotypes are indexed by the bitmask of mutated sites, bit `s` set when site `s` carries the mature residue.

In [ ]:
sites = [f"site_{i}" for i in range(10)]

def find_param_file(ab, ag):
    """Path to the LOGO-selected coefficients for one complex, or None if it was not fitted."""
    path = f"{data_dir}/{file_prefix[ab]}_{ag}_BEST_df_params.csv"
    return path if os.path.exists(path) else None

def load_terms(path):
    """Read df_params into (site tuple, coefficient) pairs; the intercept carries an empty tuple."""
    t = pd.read_csv(path)
    out = []
    for _, row in t.iterrows():
        term = str(row["term"]).strip()
        c = float(row["coefficient"])
        if term.startswith("site_"):
            out.append((tuple(sorted(int(p.split("_")[1]) for p in term.split(","))), c))
        else:
            out.append(((), c))
    return out

def find_kds_file(ab):
    """Path to the raw measurements for one antibody, or None if absent."""
    path = f"{data_dir}/{file_prefix[ab]}_kds.csv"
    return path if os.path.exists(path) else None

def load_observed(ab, ag):
    """Per-measurement observed -log10(Kd) plus each measurement's genotype index."""
    kf = find_kds_file(ab)
    if kf is None:
        return None, None
    k = pd.read_csv(kf)
    k = k[(k["antigen"] == ag) & k["nlog10_Kd"].notna()]
    if len(k) == 0:
        return None, None
    # bit s is set when site s carries the mature residue
    idx = ((k[sites].values == "M") * (1 << np.arange(10))).sum(axis=1)
    obs = k["nlog10_Kd"].values.astype(float)
    if variance_basis == "genotype":
        g = pd.DataFrame({"idx": idx, "obs": obs}).groupby("idx", as_index=False)["obs"].mean()
        return g["idx"].values, g["obs"].values
    return idx, obs


## Walsh-Hadamard decomposition

Reconstruct the latent landscape over all 1024 genotypes, transform to the Walsh-Hadamard basis, truncate at each order, censor, then score against observed data. `explained_by_order` floors negative cumulative R2 at zero and forces monotonicity, so an absent segment means that order added nothing.

In [ ]:
# Latent -log10(Kd) over all 1024 genotypes, indexed by the bitmask of mutated sites
def predict_latent(terms):
    n = 1 << 10
    idx = np.arange(n)
    y = np.zeros(n)
    for st, c in terms:
        if not st:
            y += c
            continue
        mask = 0
        for s in st:
            mask |= (1 << s)
        y += c * ((idx & mask) == mask)
    return y

# In-place fast Walsh-Hadamard transform; unnormalized, so divide by n for coefficient
def fwht(a):
    a = np.asarray(a, float).copy()
    n = len(a); h = 1
    while h < n:
        for i in range(0, n, h * 2):
            x = a[i:i+h].copy(); y = a[i+h:i+2*h].copy()
            a[i:i+h] = x + y
            a[i+h:i+2*h] = x - y
        h *= 2
    return a

# Raw incremental percent of observed variance explained by each interaction order
def variance_by_order(ab, ag):
    pf = find_param_file(ab, ag)
    if pf is None:
        return None
    idx, obs = load_observed(ab, ag)
    if idx is None or len(obs) == 0:
        return None
    ss_tot = float(((obs - obs.mean()) ** 2).sum())
    if ss_tot == 0:
        return None
    n = 1 << 10
    fhat = fwht(predict_latent(load_terms(pf))) / n
    order = np.array([bin(s).count("1") for s in range(n)])
    nz = np.abs(fhat) > 1e-9
    max_order = int(order[nz].max()) if nz.any() else 0
    r2 = {0: 0.0}   # observed mean is the reference, so order 1 covers the whole additive model
    for k in range(1, max_order + 1):
        pred = fwht(fhat * (order <= k))[idx]   # best order<=k approximation of the landscape
        if censor_floor is not None:
            pred = np.maximum(pred, censor_floor)
        r2[k] = 1.0 - float(((obs - pred) ** 2).sum()) / ss_tot
    inc = {k: 100.0 * (r2[k] - r2[k - 1]) for k in range(1, max_order + 1)}
    for k in range(max_order + 1, 11):
        inc[k] = 0.0
    return inc

# Display segments: cumulative R2 floored at zero and forced monotone
def explained_by_order(ab, ag):
    inc = variance_by_order(ab, ag)
    if inc is None:
        return None
    cum = np.cumsum([inc[k] for k in range(1, 11)])
    seg, prev, rmax = {}, 0.0, 0.0
    for i, k in enumerate(range(1, 11)):
        rmax = max(rmax, cum[i])
        val = max(rmax, 0.0)
        seg[k] = max(val - prev, 0.0)
        prev = val
    return seg

def censored_frac(ab, ag):
    idx, obs = load_observed(ab, ag)
    if obs is None:
        return None
    return float((obs <= censor_floor + 1e-3).mean())

def panel_orders(antibodies):
    mx = 0
    for ab in antibodies:
        for ag in antigen_order:
            pf = find_param_file(ab, ag)
            if pf is not None:
                mx = max(mx, max((len(s) for s, _ in load_terms(pf)), default=0))
    return list(range(1, max(mx, 1) + 1))

## Panel membership

A complex fails if the full model explains no more variance than the observed mean, or if more than half its measurements sit at the floor. An antibody is kept only when both of its complexes pass. Set `panel_antibodies` in the config to override.

In [ ]:
# Per-complex full-model R2 and censoring (these measures are used to decide whether to include)

def inclusion_table():
    """s."""
    rows = []
    for ab in all_antibodies:
        for ag in antigen_order:
            inc = variance_by_order(ab, ag)
            rows.append(dict(antibody=ab, antigen=ag,
                             r2_pct=None if inc is None else sum(inc[k] for k in range(1, 11)),
                             censored=censored_frac(ab, ag)))
    return pd.DataFrame(rows)

# Keep an antibody only when both of its complexes clear the R2 and censoring thresholds
def select_antibodies(tab):
    keep = []
    for ab in all_antibodies:
        sub = tab[tab.antibody == ab]
        if sub.r2_pct.isna().any() or sub.censored.isna().any():
            continue
        if sub.r2_pct.min() > min_r2_pct and sub.censored.max() <= max_censored_frac:
            keep.append(ab)
    return keep

## Figure

In [ ]:
# Blend a hex color toward white by fraction t
def shade(hexcolor, t):
    h = hexcolor.lstrip("#")
    r, g, b = (int(h[i:i+2], 16) for i in (0, 2, 4))
    return "#%02x%02x%02x" % tuple(int(round(v + (255 - v) * t)) for v in (r, g, b))

def text_on(hexcolor):
    h = hexcolor.lstrip("#")
    r, g, b = (int(h[i:i+2], 16) / 255 for i in (0, 2, 4))
    return "white" if (0.299*r + 0.587*g + 0.114*b) < 0.55 else "#222222"

class squished_patch(Patch):
    # own type so the unexplained swatch can take a narrower handler than the graded strips
    pass

class handler_squished(HandlerPatch):
    def __init__(self, frac):
        self.frac = frac
        super().__init__()

    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        return super().create_artists(legend, orig_handle, xdescent, ydescent,
                                      width * self.frac, height, fontsize, trans)

# One graded strip per antigen plus a narrower gray swatch for unexplained
def variance_handles():
    tag = "orders %d\u2192%d" % (orders_shown[0], orders_shown[-1])
    handles, labels = [], []
    for ag in antigen_order:
        # edgecolor none keeps the shades touching, so a strip reads as one graded block
        strip = tuple(Patch(facecolor=shade(antigen_base[ag], order_lighten[k]), edgecolor="none")
                      for k in orders_shown)
        handles.append(strip); labels.append(f"{ag} ({tag})")
    handles.append(squished_patch(facecolor=unexplained_color, edgecolor="none"))
    labels.append("unexplained")
    hmap = {tuple: HandlerTuple(ndivide=None, pad=legend_shade_pad),
            squished_patch: handler_squished(legend_unexplained_frac)}
    return handles, labels, hmap

def fig_variance_hbar(antibodies):
    rows = [[ab, ag, explained_by_order(ab, ag)] for ab in antibodies for ag in antigen_order]

    ypos, y, prev = [], 0.0, None
    for ab, ag, seg in rows:
        if prev is not None and ab != prev:
            y += group_gap
        ypos.append(y); y += 1.0; prev = ab
    top = max(ypos)
    ypos = [top - v for v in ypos]

    span = len(rows) + group_gap * (len(antibodies) - 1)
    fig_h = 0.32 * span + 2.0
    fig, ax = plt.subplots(figsize=(plot_aspect * fig_h, fig_h))

    for (ab, ag, seg), yy in zip(rows, ypos):
        if seg is None:
            ax.text(50, yy, "pending", ha="center", va="center", fontsize=fs_seg,
                    style="italic", color="#888888")
            continue
        left = 0.0
        for k in orders_shown:
            hval = seg[k]
            if hval > 0:
                col = shade(antigen_base[ag], order_lighten[k])
                ax.barh(yy, hval, left=left, height=0.72, color=col,
                        edgecolor="white", linewidth=0.6)
                if bar_labels != "none" and hval >= min_seg_label:
                    ax.text(left + hval / 2, yy, f"{hval:.1f}", ha="center", va="center",
                            fontsize=fs_seg, color=text_on(col))
            left += hval
        rest = max(0.0, 100.0 - left)
        ax.barh(yy, rest, left=left, height=0.72, color=unexplained_color,
                edgecolor="white", linewidth=0.6)
        if bar_labels == "all" and rest >= min_seg_label:
            ax.text(left + rest / 2, yy, f"{rest:.1f}", ha="center", va="center",
                    fontsize=fs_seg, color=text_on(unexplained_color))

    ax.set_yticks([])   # antigen is carried by hue, so tick labels would repeat the legend
    centers = {}
    for (ab, ag, seg), yy in zip(rows, ypos):
        centers.setdefault(ab, []).append(yy)
    for ab, yl in centers.items():
        ax.text(-0.02, sum(yl) / len(yl), ab, transform=ax.get_yaxis_transform(),
                ha="right", va="center", fontsize=fs_header, fontweight="semibold")

    ax.set_xlim(0, 100)
    ax.set_ylim(-0.9, top + 0.9)
    ax.set_xticks(np.arange(0, 101, 10))
    ax.set_xlabel("% of observed variance", fontsize=fs_axis)
    ax.tick_params(labelsize=fs_tick, length=4)
    if pct_gridlines:
        for xv in range(10, 100, 10):
            ax.axvline(xv, color="#808080", alpha=0.35, linewidth=0.6, zorder=5)

    handles, labels, hmap = variance_handles()
    fig.legend(handles, labels, handler_map=hmap, loc="upper center", bbox_to_anchor=(0.5, 0.995),
               ncol=3, frameon=False, fontsize=fs_legend, handlelength=2.4,
               columnspacing=1.2, handleheight=2.0)
    fig.subplots_adjust(left=0.18, right=0.98, top=0.90, bottom=0.06)
    return fig


## Build

Prints the inclusion table, resolves the panel, then writes PNG, PDF and SVG.

In [ ]:
tab = inclusion_table()
print(tab.round(3).to_string(index=False))

antibodies = panel_antibodies or select_antibodies(tab)
print("\npanel:", ", ".join(antibodies))
print("dropped:", ", ".join(a for a in all_antibodies if a not in antibodies))

if orders_shown is None:
    orders_shown = panel_orders(antibodies)
print("orders shown:", orders_shown)

fig = fig_variance_hbar(antibodies)
for ext in save_exts:
    path = os.path.join(fig_dir, f"{fig_stem}.{ext}")
    fig.savefig(path, dpi=fig_dpi if ext == "png" else None,
                bbox_inches="tight", facecolor="white")
    print("wrote", path)

svg = os.path.join(fig_dir, f"{fig_stem}.svg")
if os.path.exists(svg):
    assert open(svg).read().count("<text") == 0, "live text in SVG, check svg.fonttype"

plt.show()
